# ME 313 · Lab 7.1 — Read temperature from a thermal spectrum
**Module 7 companion (Radiation).**

A hot surface's emission spectrum encodes its temperature (Planck's law; the peak obeys Wien's $\lambda_{max}T=2898\,\mu m\cdot K$). 
Inverting a measured spectrum to get $T$ is the basis of **IR thermometry and remote sensing** — and it's the same two-spectra idea behind the car greenhouse example. 
You generate a noisy spectrum, recover $T$ two ways (full Planck fit vs. Wien's single-peak shortcut), and compare.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit
C1, C2 = 3.7418e8, 1.4388e4   # radiation constants (wavelength in micrometres)

### 1. A measured emission spectrum
Planck spectral emissive power $E_{\lambda,b}=C_1/\{\lambda^5[\exp(C_2/\lambda T)-1]\}$, with 4% noise.


In [ ]:
rng = np.random.default_rng(1)
def planck(lam, T, scale):
    return scale * C1 / (lam**5 * (np.exp(C2/(lam*T)) - 1))

T_true = 1200.0                       # the unknown we will recover (K)
lam = np.linspace(1, 15, 60)          # micrometres
E = planck(lam, T_true, 1.0)
E_meas = E * (1 + rng.normal(0, 0.04, lam.shape))
print('spectrum sampled from 1 to 15 um')

### 2a. Recover $T$ by fitting the full Planck curve


In [ ]:
(T_fit, scale_fit), _ = curve_fit(planck, lam, E_meas, p0=[800, 1.0], maxfev=10000)
print(f'Planck-fit T = {T_fit:.0f} K   (true {T_true:.0f})')

### 2b. Recover $T$ from Wien's peak (the shortcut)


In [ ]:
lam_peak = lam[np.argmax(E_meas)]
T_wien = 2898.0/lam_peak
print(f'Wien-peak T  = {T_wien:.0f} K   (peak at {lam_peak:.2f} um)')
print('Note: the full fit uses the whole curve and beats the single-point Wien estimate,')
print('especially when the sampling is coarse or the true peak falls between samples.')

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(lam, E_meas, 'o', ms=4, label='measured')
plt.plot(lam, planck(lam, T_fit, scale_fit), '-', label=f'Planck fit ({T_fit:.0f} K)')
plt.axvline(2898/T_true, ls=':', color='grey', label='true Wien peak')
plt.xlabel('wavelength (um)'); plt.ylabel('E_lambda (arb.)'); plt.legend(); plt.grid(alpha=.3); plt.show()

### Your turn
1. Change `T_true` to 900 K and 2500 K. Watch the peak slide (Wien) and confirm both methods track it.
2. Restrict the sampled band to 8–14 um (a real LWIR camera). Does the Planck fit still work? Does Wien?
3. **AI companion:** ask an LLM for the temperature given the peak wavelength, then check it against Wien and your fit.
4. Connect to the car: plot Planck curves at 5800 K and 333 K on one axis and mark glass's 2.7 um cutoff.
